# Add Character History

The query below will add the previous updates for characters based on their current stats. This only needs to be run when doing a clean load of the database and then any changes will need to be made to the historical records. It just really populates the effective date and update codes.

In [0]:
WITH 
one_update
  (
    --Characters with Update
    SELECT
        c.character_id
      ,c.character_name
    FROM
      mcp.game.characters c
      INNER JOIN mcp.game.character_updates cu
        ON c.character_id = cu.character_id
    GROUP BY 
      ALL
    HAVING 
      COUNT(cu.character_id) = 1
  )
,one_update_release
  (
    SELECT
       c.character_id
      ,c.character_name
      ,p.release_date
      ,pc.sku
      ,ROW_NUMBER() OVER (PARTITION BY c.character_id ORDER BY p.release_date ASC) row_no
    FROM
      one_update c
      INNER JOIN mcp.game.product_to_character pc
        ON c.character_id = pc.character_id
      INNER JOIN mcp.game.product_codes p
        ON pc.sku = p.sku
  )
,multi_update
  (
    SELECT
       c.character_id
      ,c.character_name
      ,c.effective_date
      ,c.update_key
      ,COUNT(cu.update_key) update_count
    FROM
      mcp.game.characters c
      INNER JOIN mcp.game.character_updates cu
        ON c.character_id = cu.character_id
    GROUP BY 
      ALL
    HAVING 
      COUNT(cu.character_id) > 1
  )
,multi_first
  (
    SELECT 
      m.character_id
      ,m.character_name
      ,au.update_key
      ,au.effective_date
    FROM 
      multi_update m
      INNER JOIN mcp.game.character_updates cu
        ON m.character_id = cu.character_id
      INNER JOIN mcp.game.amg_updates au
        ON cu.update_key = au.update_key
        AND m.effective_date > au.effective_date
  )
,multi_release
  (
    SELECT
       c.character_id
      ,c.character_name
      ,p.release_date
      ,pc.sku
      ,ROW_NUMBER() OVER (PARTITION BY c.character_id ORDER BY p.release_date ASC) row_no
    FROM
      multi_update c
      INNER JOIN mcp.game.product_to_character pc
        ON c.character_id = pc.character_id
      INNER JOIN mcp.game.product_codes p
        ON pc.sku = p.sku
  )
INSERT INTO mcp.game.characters
  (
     character_id
    ,character_name
    ,alter_ego
    ,threat
    ,front_health
    ,back_health
    ,extra_power
    ,leadership_affiliation_id
    ,leadership_card_id
    ,gem_bearer
    ,gem_limit
    ,effective_date
    ,update_key
    ,current_version
  )
SELECT
   c.character_id
  ,c.character_name
  ,c.alter_ego
  ,c.threat
  ,c.front_health
  ,c.back_health
  ,c.extra_power
  ,c.leadership_affiliation_id
  ,c.leadership_card_id
  ,c.gem_bearer
  ,c.gem_limit
  ,mf.effective_date
  ,mf.update_key update_key
  ,false current_version
FROM
  multi_first mf
  INNER JOIN mcp.game.characters c
    ON mf.character_id = c.character_id

UNION ALL

SELECT
   c.character_id
  ,c.character_name
  ,c.alter_ego
  ,c.threat
  ,c.front_health
  ,c.back_health
  ,c.extra_power
  ,c.leadership_affiliation_id
  ,c.leadership_card_id
  ,c.gem_bearer
  ,c.gem_limit
  ,mr.release_date effective_date
  ,NULL update_key
  ,false current_version
FROM
  multi_release mr
  INNER JOIN mcp.game.characters c
    ON mr.character_id = c.character_id
WHERE
  mr.row_no = 1

UNION ALL

SELECT
   c.character_id
  ,c.character_name
  ,c.alter_ego
  ,c.threat
  ,c.front_health
  ,c.back_health
  ,c.extra_power
  ,c.leadership_affiliation_id
  ,c.leadership_card_id
  ,c.gem_bearer
  ,c.gem_limit
  ,r.release_date effective_date
  ,NULL update_key
  ,false current_version
FROM
  one_update_release r
  INNER JOIN mcp.game.characters c
    ON r.character_id = c.character_id
WHERE
  r.row_no = 1

In [0]:
SELECT
  c.character_id
  ,c.update_key
  ,c.effective_date
  ,c.character_name
  ,c.threat
  ,c.front_health
  ,c.back_health
FROM
  mcp.game.characters c
WHERE
  NOT c.current_version
ORDER BY
   c.character_name
  ,c.effective_date DESC

In [0]:
--Black Bolt
UPDATE mcp.game.characters
SET
  front_health = 5
WHERE
  character_id = '00340101'
  AND effective_date = '2021-02-11'
;
--Black Widow 2
UPDATE mcp.game.characters
SET
  front_health = 5
  ,back_health = 5
WHERE
  character_id = '00240101'
  AND effective_date = '2020-05-06'
;
--Bullseye
UPDATE mcp.game.characters
SET
  front_health = 5
  ,back_health = 5
  ,threat = 3
WHERE
  character_id = '00300101'
  AND effective_date = '2020-10-22'
;
--Cable
UPDATE mcp.game.characters
SET
  front_health = 6
WHERE
  character_id = '00470101'
  AND effective_date = '2021-06-09'
;
--Carnage
UPDATE mcp.game.characters
SET
  front_health = 7
WHERE
  character_id = '00500101'
  AND effective_date = '2021-06-23'
;
--Casandra Nova
UPDATE mcp.game.characters
SET
  back_health = 7
WHERE
  character_id = '00530101'
  AND effective_date = '2021-07-07'
;
--Clea
UPDATE mcp.game.characters
SET
  front_health = 6
WHERE
  character_id = '00670101'
  AND effective_date = '2021-09-08'
;
--Dr. Strange
UPDATE mcp.game.characters
SET
  back_health = 7
WHERE
  character_id = '00230101'
  AND effective_date = '2020-09-10'
;
--Ebony Maw
UPDATE mcp.game.characters
SET
  back_health = 4
WHERE
  character_id = '00190102'
  AND effective_date = '2020-06-28'
;
--Gamora
UPDATE mcp.game.characters
SET
   front_health = 5
  ,back_health = 5
WHERE
  character_id = '00160101'
  AND effective_date = '2020-04-23'
;
--Ghost Rider
UPDATE mcp.game.characters
SET
   front_health = 7
  ,back_health = 6
WHERE
  character_id = '00270101'
  AND effective_date = '2020-09-11'
;
--Hawkeye
UPDATE mcp.game.characters
SET
   front_health = 4
  ,back_health = 5
WHERE
  character_id = '00240102'
  AND effective_date = '2020-05-06'
;
--Magik
UPDATE mcp.game.characters
SET
   front_health = 5
WHERE
  character_id = '00570102'
  AND effective_date = '2022-02-10'
;
--Malekith the Accursed
UPDATE mcp.game.characters
SET
   front_health = 10
  ,back_health = 8
WHERE
  character_id = '00930101'
  AND effective_date = '2022-08-11'
;
--Nick Fury
UPDATE mcp.game.characters
SET
   front_health = 6
  ,back_health = 5
WHERE
  character_id = '00550101'
  AND effective_date = '2022-03-25'
;
--Professor X
UPDATE mcp.game.characters
SET
  back_health = 5
WHERE
  character_id = '01510101'
  AND effective_date = '2024-02-29'
;
--Psylocke
UPDATE mcp.game.characters
SET
  back_health = 5
WHERE
  character_id = '01030102'
  AND effective_date = '2023-03-09'
;
--Psylocke
UPDATE mcp.game.characters
SET
   front_health = 6
  ,back_health = 6
WHERE
  character_id = '00200102'
  AND effective_date = '2020-06-10'
;
--Sentinel MK4
UPDATE mcp.game.characters
SET
   front_health = 7
  ,back_health = 6
WHERE
  character_id IN ('00510102','00510101')
  AND effective_date = '2022-10-13'
;
--Sentinel Prime MK4
UPDATE mcp.game.characters
SET
   front_health = 10
  ,back_health = 8
WHERE
  character_id = '01600101'
  AND effective_date = '2022-10-13'
;
--Star Lord
UPDATE mcp.game.characters
SET
  back_health = 5
WHERE
  character_id = '00180101'
  AND effective_date IN ('2021-11-30','2020-03-12')
;
--Thor, Prince of Asgard
UPDATE mcp.game.characters
SET
   front_health = 6
  ,back_health = 8
WHERE
  character_id = '00110101'
  AND effective_date = '2020-01-30'
;
--Toad
UPDATE mcp.game.characters
SET
   front_health = 5
  ,back_health = 3
WHERE
  character_id = '00420102'
  AND effective_date = '2020-11-12'
;
--Wolverine
UPDATE mcp.game.characters
SET
   front_health = 7
  ,back_health = 5
WHERE
  character_id = '00400102'
  AND effective_date = '2020-11-12'
;
--Yondu
UPDATE mcp.game.characters
SET
   front_health = 6
  ,back_health = 5
WHERE
  character_id = '01260102'
  AND effective_date = '2025-01-30'
;

In [0]:
INSERT INTO mcp.game.product_to_character
  (
     character_id
    ,sku
  )
VALUES
  ('01580101','CP158')
  ,('20130101','CA13')
  ,('10040101','CPE04')
  ,('10040102','CPE04')
  ,('10030101','CPE03')
  ,('20230101','CA23')
  ,('20230102','CA23')
  ,('10030102','CPE03')